# Example 6.2

The reference surface is the unit sphere. For $p\in M$,
    $r(t,p)=((1+tq(p_z))p_x,(1+tq(p_z))p_y,p_z)$,
    where $q(s)=-0.32+0.48(1-s^2)^2$.
    Initial values use nodal Lagrange interpolation; the background Regge metric
    comes from the exact sphere. The solver metric rate uses the FE endpoint derivatives.

Run all cells in order in a Python kernel with NGSolve and NumPy. All required functions are included here. After changing parameters, restart and run all.


## 1. Parameters, mesh and finite-element spaces


In [ ]:
import numpy as np
from ngsolve import *
from netgen.occ import Pnt, Axis, Face, Wire, SplineApproximation, OCCGeometry, Z
from netgen.meshing import MeshingStep

# Original parameters. For a quick run: h=0.5, r=k=1.
h = 0.1                 # Maximum mesh size
r = 6                   # Regge metric degree
k = 6                   # H1 embedding degree
tau = 0.025
T = 1.0
num_threads_asm = 4
show_surface = False    # Optional ngsolve.webgui display in the last cell
inverse = "pardiso"      # Use "umfpack" if provided instead by your NGSolve build
Ax, Ay, Az = 1, 1, 1      # Fixed reference surface for this example
metric_bonus_intorder = 10
quad_order_asm = 30

if not (np.isfinite(tau) and np.isfinite(T) and tau > 0 and T > 0):
    raise ValueError("tau and T must be finite and positive")
N_steps = int(round(T / tau))
if N_steps < 3 or not np.isclose(N_steps*tau, T, rtol=1e-12, atol=0):
    raise ValueError("BDF3 requires integer T/tau >= 3")
tau = T / N_steps
SetNumThreads(num_threads_asm)

# Retain the original spline-of-revolution surface mesh.
points_on_curve = [Pnt(0, Ay*np.sin(phi), Az*np.cos(phi))
                   for phi in np.linspace(0, np.pi, 200)]
face = Face(Wire([SplineApproximation(points_on_curve, tol=1e-12)]))
surface = face.Revolve(Axis((0, 0, 0), Z), 360)
mesh = Mesh(OCCGeometry(surface).GenerateMesh(
    maxh=h, perfstepsend=MeshingStep.MESHSURFACE, grading=0.5))
surface_region = mesh.Boundaries('.*')
ds_asm = ds(intrules={ET.TRIG: IntegrationRule(ET.TRIG, order=quad_order_asm)})
fesG = HCurlCurl(mesh, order=r)
fesC = NumberSpace(mesh)
fesV = VectorH1(mesh, order=k)
fes = H1(mesh, order=k)
fesMix = fesV*fesC*fesC*fesC*fesC*fesC*fesC
print(f"Embedding DOFs: {fesV.ndof}; time steps: {N_steps}")


## 2. Reference geometry and initialization

Point-value interpolation in the oriented H1 basis.


In [ ]:
def interpolate_surface_lagrange(fes, evaluate, order):
    """Interpolate point values in the oriented triangular H1 basis."""
    if not isinstance(order, (int, np.integer)) or order < 1:
        raise ValueError('order must be a positive integer')
    mesh = fes.mesh
    if (mesh.dim != 3 or mesh.ne != 0 or mesh.GetCurveOrder() != 1
            or mesh.deformation is not None):
        raise ValueError('expected an undeformed, piecewise-flat surface mesh in R^3')
    if fes.type != 'h1ho' or fes.dim != 1:
        raise ValueError('fes must be a scalar H1 space')

    nodes = [(i / order, j / order)
             for i in range(order + 1) for j in range(order + 1 - i)]

    ir = IntegrationRule(nodes, [1.0] * len(nodes))
    coords = CF((x, y, z))
    elements = []
    physical_points = []
    matrices = {}
    for element in mesh.Elements(BND):
        if element.type != ET.TRIG:
            raise ValueError('only triangular surface elements are supported')
        fe = fes.GetFE(element)
        dofs = np.asarray(fes.GetDofNrs(element), dtype=int)
        if fe.order != order or fe.ndof != len(nodes) or np.any(dofs < 0):
            raise ValueError('expected a uniform, unconstrained local P_k space')
        matrix = np.array([fe.CalcShape(u, v).NumPy().copy() for u, v in nodes])

        key = matrix.tobytes()
        if key not in matrices:
            matrices[key] = matrix
        elements.append((dofs, key))
        physical_points.append(np.asarray(coords(mesh.GetTrafo(element)(ir))))
    if not elements:
        raise ValueError('the surface mesh is empty')

    values = np.asarray(evaluate(np.concatenate(physical_points)), dtype=float)
    if values.ndim == 1:
        values = values[:, None]
    npoints = len(elements) * len(nodes)
    if (values.ndim != 2 or values.shape[0] != npoints or values.shape[1] == 0
            or not np.isfinite(values).all()):
        raise ValueError('evaluate must return finite (N,) or (N, d) values')
    values = values.reshape(len(elements), len(nodes), -1)
    coefficients = np.zeros((fes.ndof, values.shape[2]))
    counts = np.zeros(fes.ndof, dtype=int)
    for (dofs, key), local_values in zip(elements, values):
        local_coefficients = np.linalg.solve(matrices[key], local_values)
        np.add.at(coefficients, dofs, local_coefficients)
        np.add.at(counts, dofs, 1)
    if np.any(counts == 0):
        raise ValueError('fes contains DOFs outside the triangular surface')
    coefficients /= counts[:, None]

    max_error = max(float(np.max(np.abs(matrices[key] @ coefficients[dofs] - local_values)))
                    for (dofs, key), local_values in zip(elements, values))
    scale = max(1.0, float(np.max(np.abs(values))))
    if not np.isfinite(coefficients).all() or max_error > 1e-11 * scale:
        raise RuntimeError('nodal interpolation failed: inconsistent traces or ill-conditioned basis')
    return coefficients, {
        'sample_points': npoints,
        'max_nodal_error': max_error,
    }


In [ ]:
rho_geometry = sqrt(x*x + y*y + z*z)
a_geometry = CF((x, y, z)) / rho_geometry
Da_geometry = (Id(3) - OuterProduct(a_geometry, a_geometry)) / rho_geometry
q_geometry = -0.32 + 0.48*(1 - a_geometry[2]**2)**2
dq_geometry = -1.92*a_geometry[2]*(1 - a_geometry[2]**2)

Q_geometry = CF((
    q_geometry, 0, a_geometry[0]*dq_geometry,
    0, q_geometry, a_geometry[1]*dq_geometry,
    0, 0, 0,
), dims=(3, 3))

mesh.UnsetDeformation()
def exact_endpoint_values(node_coords):
    p = node_coords / np.linalg.norm(node_coords, axis=1)[:, None]
    q = -0.32 + 0.48*(1 - p[:, 2]**2)**2
    r1 = p.copy()
    r1[:, :2] *= (1 + q)[:, None]
    return np.column_stack((p, r1))

endpoint_coeffs, initial_interp_info = interpolate_surface_lagrange(fes, exact_endpoint_values, k)
Xh0  = GridFunction(fesV)
Xh1 = GridFunction(fesV)
for i in range(3):
    Xh0.components[i].vec.FV().NumPy()[:] = endpoint_coeffs[:, i]
    Xh1.components[i].vec.FV().NumPy()[:] = endpoint_coeffs[:, i+3]
print(f"Lagrange endpoint interpolation: max nodal error = {initial_interp_info['max_nodal_error']:.3e}")
Exa_ptr = GridFunction(fesV)
Exa_ptr.vec.data = Xh1.vec - Xh0.vec

nsurf = specialcf.normal(3)
NN = OuterProduct(nsurf, nsurf)
Ptau = Id(3) - NN
J0_geometry = Da_geometry * Ptau
Jvelocity_geometry = Q_geometry * J0_geometry
J = J0_geometry
G = J.trans * J
gMh = GridFunction(fesG)
gMh.Set(
    G, definedon=mesh.Boundaries('.*'),
    dual=True,
    bonus_intorder=metric_bonus_intorder,
)
g_inv = Ptau * Inv(gMh + NN) * Ptau
sq_det = sqrt(Det(gMh + NN))

Pos0 = GridFunction(fesV)
Pos1 = GridFunction(fesV)
Pos_1 = GridFunction(fesV)
Pos_2 = GridFunction(fesV)
Pos_hat = GridFunction(fesV)

Solution= GridFunction(fesMix)

told = 0.0
Pos_2.vec.data = Xh0.vec
told = tau
Pos_1.vec.data = (1 - told)*Xh0.vec + told*Xh1.vec
told = 2 * tau
step_n = 2
Pos0.vec.data = (1 - told)*Xh0.vec + told*Xh1.vec


## 3. BDF3 time stepping

Three startup values are prescribed at $0,\tau,2\tau$. Six multipliers impose the rotational and translational constraints. Each step reassembles the matrix using the extrapolated embedding.


In [ ]:
SetNumThreads(num_threads_asm)

Exa_ptg = GridFunction(fesG)
v, alp1, alp2, alp3, beta1, beta2, beta3 = fesMix.TrialFunction()
w, alp1_t, alp2_t, alp3_t, beta1_t, beta2_t, beta3_t = fesMix.TestFunction()

Jhat_cached = CacheCF(grad(Pos_hat).Trace())
g_inv_cached = CacheCF(g_inv)
mu_cached = CacheCF(sq_det)

dv = grad(v).Trace()
dw = grad(w).Trace()
strain_v = 0.5 * (Jhat_cached.trans * dv + dv.trans * Jhat_cached)
strain_w = 0.5 * (Jhat_cached.trans * dw + dw.trans * Jhat_cached)

lhs = BilinearForm(fesMix)
lhs += (
    2 * Trace(g_inv_cached * strain_v * g_inv_cached * strain_w)
    * mu_cached * ds_asm
)

lhs += (
    InnerProduct(w, Cross((alp1, alp2, alp3), Pos_hat)
                 + (beta1, beta2, beta3)) * sq_det
).Compile() * ds_asm
lhs += (
    InnerProduct(v, Cross((alp1_t, alp2_t, alp3_t), Pos_hat)
                 + (beta1_t, beta2_t, beta3_t)) * sq_det
).Compile() * ds_asm

Jhat_rhs = grad(Pos_hat).Trace()
strain_w_rhs = 0.5 * (Jhat_rhs.trans * dw + dw.trans * Jhat_rhs)
rhs = LinearForm(fesMix)
rhs += (
    Trace(g_inv * Exa_ptg * g_inv * strain_w_rhs) * sq_det
).Compile() * ds_asm

J0_rhs = grad(Xh0).Trace()
Jdelta_rhs = grad(Xh1).Trace() - J0_rhs
Jvelocity_rhs = grad(Exa_ptr).Trace()
F0_ptg = J0_rhs.trans * Jvelocity_rhs + Jvelocity_rhs.trans * J0_rhs
F1_ptg = Jdelta_rhs.trans * Jvelocity_rhs + Jvelocity_rhs.trans * Jdelta_rhs
ptg_const = GridFunction(fesG)
ptg_slope = GridFunction(fesG)

with TaskManager():
    ptg_const.Set(F0_ptg, definedon=surface_region, dual=True,
                  bonus_intorder=metric_bonus_intorder)
    ptg_slope.Set(F1_ptg, definedon=surface_region, dual=True,
                  bonus_intorder=metric_bonus_intorder)

    # Exact-target norm on M and discrete-target norm on Mh are distinct diagnostics.
    metric_time = Parameter(0.0)
    metric_field = GridFunction(fesV)
    G0_exact = J0_geometry.trans * J0_geometry
    H_exact = Ptau * Inv(G0_exact + NN) * Ptau
    mu_exact = sqrt(Det(G0_exact + NN))
    Jt = J0_geometry + metric_time * Jvelocity_geometry
    Je = grad(metric_field).Trace() - Jt
    exact_defect = Jt.trans*Je + Je.trans*Jt + Je.trans*Je
    initial_discrete_metric = GridFunction(fesG)
    initial_discrete_metric.Set(grad(Xh0).Trace().trans * grad(Xh0).Trace(),
                               definedon=surface_region, dual=True,
                               bonus_intorder=metric_bonus_intorder)
    discrete_target = (initial_discrete_metric + metric_time*ptg_const
                       + 0.5*metric_time*metric_time*ptg_slope)
    Jh = grad(metric_field).Trace()
    discrete_defect = Jh.trans*Jh - discrete_target
    metric_densities = CF((Trace(H_exact*exact_defect*H_exact*exact_defect)*mu_exact,
                           Trace(g_inv*discrete_defect*g_inv*discrete_defect)*sq_det))
    metric_history = []

    def record_metrics(step, time, field):
        metric_time.Set(time)
        metric_field.vec.data = field.vec
        squared = np.asarray(Integrate(metric_densities, mesh,
                                       definedon=surface_region, order=40))
        if not np.isfinite(squared).all() or np.any(squared < 0):
            raise RuntimeError("Invalid metric defect integral")
        metric_history.append((step, time, *np.sqrt(squared)))

    for step, field in enumerate((Xh0, Pos_1, Pos0)):
        record_metrics(step, step*tau, field)

    for n in range(step_n + 1, N_steps + 1):
        told = T if n == N_steps else n * tau

        Exa_ptg.vec.data = ptg_const.vec + told * ptg_slope.vec

        Pos_hat.vec.data = BaseVector(
            3 * Pos0.vec.FV().NumPy()
            - 3 * Pos_1.vec.FV().NumPy()
            + Pos_2.vec.FV().NumPy()
        )

        lhs.Assemble()
        rhs.Assemble()
        Solution.vec.data = lhs.mat.Inverse(inverse=inverse) * rhs.vec

        Pos1.vec.data = BaseVector(
            6 / 11 * (
                3 * Pos0.vec.FV().NumPy()
                - 3 / 2 * Pos_1.vec.FV().NumPy()
                + 1 / 3 * Pos_2.vec.FV().NumPy()
                + tau * Solution.components[0].vec.FV().NumPy()
            )
        )
        Pos_2.vec[:] = Pos_1.vec[:]
        Pos_1.vec[:] = Pos0.vec[:]
        Pos0.vec[:] = Pos1.vec[:]
        step_n = n
        record_metrics(n, told, Pos1)
        print(f"step {n}/{N_steps}, t={told:.6f}")


## 4. Errors

The graph error and exact metric defect use the fixed exact reference-surface metric. The discrete metric defect uses the Regge target and fixed discrete background metric. Maxima include every time level, including all three startup values. The CSV in `results/` is replaced when the same parameters are run again.


In [ ]:
import ngsolve as ng
mesh.UnsetDeformation()

rho_exact = ng.sqrt(ng.x**2 + ng.y**2 + ng.z**2)
p_exact = ng.CF((ng.x/rho_exact, ng.y/rho_exact, ng.z/rho_exact))
Da_cf = (ng.Id(3) - ng.OuterProduct(p_exact, p_exact)) / rho_exact
nh = ng.specialcf.normal(3)
Nh = ng.OuterProduct(nh, nh)
Ph = ng.Id(3) - Nh
J0_exact = Da_cf * Ph

q_exact = -0.32 + 0.48*(1 - p_exact[2]**2)**2
dq_exact = -1.92*p_exact[2]*(1 - p_exact[2]**2)
Q_exact = ng.CF((
    q_exact, 0, p_exact[0]*dq_exact,
    0, q_exact, p_exact[1]*dq_exact,
    0, 0, 0,
), dims=(3, 3))
r_minus_l = ng.CF((
    (1 + T*q_exact)*p_exact[0],
    (1 + T*q_exact)*p_exact[1],
    p_exact[2],
))
Jr_exact = (ng.Id(3) + T*Q_exact) * J0_exact

e = Pos1 - r_minus_l
Jh = ng.grad(Pos1).Trace()
Je = Jh - Jr_exact
S = 0.5*(Jr_exact.trans*Je + Je.trans*Jr_exact)

metric_defect = 2*S + Je.trans*Je

G0_exact = J0_exact.trans * J0_exact
H_exact = Ph * ng.Inv(G0_exact + Nh) * Ph
mu_exact = ng.sqrt(ng.Det(G0_exact + Nh))

J_initial = ng.grad(Xh0).Trace()
Je_initial = J_initial - J0_exact

metric_defect_initial = (J0_exact.trans*Je_initial
                         + Je_initial.trans*J0_exact
                         + Je_initial.trans*Je_initial)

error_densities = ng.CF((
    ng.InnerProduct(e, e),
    ng.Trace(H_exact*S*H_exact*S),
    ng.Trace(H_exact*metric_defect*H_exact*metric_defect),
    ng.Trace(H_exact*metric_defect_initial*H_exact*metric_defect_initial),
)) * mu_exact
quad_order_err = 40
_squared_errors = np.asarray(ng.Integrate(
    error_densities, mesh, definedon=mesh.Boundaries('.*'), order=quad_order_err,
), dtype=float)
L2_sq, strain_sq, metric_sq, initial_metric_sq = map(float, _squared_errors)

err_graph = float(np.sqrt(L2_sq + strain_sq))
err_metric = float(np.sqrt(metric_sq))
err_metric_initial = float(np.sqrt(initial_metric_sq))

history = np.asarray(metric_history)
if len(history) != N_steps + 1 or not np.allclose(
        history[[0, -1], 2], [err_metric_initial, err_metric], rtol=1e-9, atol=1e-13):
    raise RuntimeError("Metric history is incomplete or disagrees with endpoint errors")
print(f"Graph error: {err_graph:.12e}")
print(f"Exact metric defect: initial={err_metric_initial:.12e}, final={err_metric:.12e}")
for column, name in ((2, "Exact"), (3, "Discrete")):
    peak = int(np.argmax(history[:, column]))
    print(f"{name} metric defect: max={history[peak, column]:.12e} at t={history[peak, 1]:.6f}")
from pathlib import Path
result_dir = Path('results') / f'example_6_2_h{h:g}_r{r}_k{k}_tau{tau:g}_T{T:g}'
result_dir.mkdir(parents=True, exist_ok=True)
np.savetxt(result_dir / 'metric_history.csv', history, delimiter=',',
           header='step,time,exact_metricerr,discrete_metricerr', comments='')


## 5. Optional surface view

Set `show_surface=True` to enable the Jupyter web widget.


In [ ]:
if show_surface:
    from ngsolve.webgui import Draw
    mesh.UnsetDeformation()
    deform = GridFunction(fesV)
    deform.Set(Pos1 - CF((x, y, z)), definedon=surface_region)
    mesh.SetDeformation(deform)
    Draw(mesh)
    mesh.UnsetDeformation()
